# NB33: wide pretrained graph model with interval loss

Select **L4**, then **Run all**. Mount **your Google account** Drive. All five folds run serially, with best and resumable-last checkpoints saved each epoch. Rerun all after disconnection: completed folds skip and interrupted folds resume. Keep RUN unchanged when resuming; changing code, dependencies or data is rejected.

This independently trains the public [Scigantic wide CheMeleon interval recipe](https://github.com/Scigantic/open-admet-challenge-cyp/tree/main/training), with our audited training table and frozen scaffold splits. It uses the general CheMeleon foundation, not an organizer CYP model or competitor predictions. Four scored interval-loss heads plus 18 auxiliary MAE heads, all uniformly weighted. Unlike our earlier 700-wide NB28/29, it uses 2048 hidden units, depth4, and pretrained weights. The published ensemble result does not establish that this individual model will reach 0.44.

All challenge labels/bounds verified against our data; holdout scaffolds and blind connectivity removed. Four existing challenge connectivity duplicates stay within their original fold. Source code MIT-0; public auxiliary data retain upstream attribution. Code uses float32 with TF32 permitted, batch128, up to200 epochs and validation patience30. Data grouping, holdout purge and folds differ from the source; no claimed exact replication.

Development selection: weights0,.1,.2,.4,.6, at least4/5 improving folds, and combined macro ST-RAE gain≥.005. Holdout metrics are revealed only after freezing a passing choice. Existing development/holdout reuse limits confidence. No leaderboard score forecast or automatic submission CSV. **NB33 is a notebook number, not submission v33.**


In [ ]:
RUN='a'
from google.colab import drive
drive.mount('/content/drive')
import subprocess,sys,os
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
subprocess.run([sys.executable,'-m','pip','install','-q','chemprop==2.3.1','lightning==2.6.6'],check=True)


In [ ]:
from pathlib import Path
import json,hashlib,zipfile,shutil,time
import torch
assert torch.cuda.is_available(), 'Select an L4 GPU runtime first.'
print('GPU:',torch.cuda.get_device_name(0))
CACHE=Path('/content/nb33_cache');CACHE.mkdir(exist_ok=True)
PAYLOAD=Path('/content/drive/MyDrive/cyp_payloads/nb33_wide_4bcc5edc6537.zip')
assert PAYLOAD.exists(), 'Payload missing: mount your Google account Drive.'
assert hashlib.sha256(PAYLOAD.read_bytes()).hexdigest()=='4bcc5edc653794ca755c2c58e2fb266266dd7d60b79888bc69c0352c4b3b4698'
with zipfile.ZipFile(PAYLOAD) as z:
    for n in z.namelist():assert not Path(n).is_absolute() and '..' not in Path(n).parts
    z.extractall(CACHE)
manifest=json.loads((CACHE/'manifest.json').read_text())
for n,digest in manifest.items():assert hashlib.sha256((CACHE/n).read_bytes()).hexdigest()==digest,n
OUT=Path('/content/drive/MyDrive/cyp_results')/f'nb33_{RUN}_wide_interval';OUT.mkdir(parents=True,exist_ok=True)
for source in (CACHE/'inputs').iterdir():
    dest=OUT/source.name
    if dest.exists():assert dest.read_bytes()==source.read_bytes(), 'Changed inputs: choose a new RUN.'
    else:shutil.copy2(source,dest)
os.chdir(CACHE);os.environ.update(NB33_OUT=str(OUT),NB33_DEVICE='gpu',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2')
print('Data hashes verified; outputs:',OUT)
print(json.loads((OUT/'data_audit.json').read_text()))


In [ ]:
# Finite five-fold run. Each fold is a separate process to release GPU memory.
started=time.time()
for fold in range(5):
    print(f'\nFOLD {fold}/4: starting or resuming',flush=True)
    with (OUT/f'fold{fold}.log').open('a') as log:
        proc=subprocess.Popen([sys.executable,'-u','train_nb33.py','--fold',str(fold),'--batch-size','128'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            rc=proc.wait()
        except BaseException:
            proc.terminate();proc.wait();raise
    assert rc==0, f'Fold {fold} stopped. Read its log; rerun to resume from last saved epoch.'
    subprocess.run([sys.executable,'eval_nb33.py'],check=True)
print('All folds finished in',round((time.time()-started)/3600,2),'hours')


In [ ]:
# The holdout script freezes the passing development choice before scoring.
subprocess.run([sys.executable,'holdout_nb33.py'],check=True)
summary=json.loads((OUT/'development_results.json').read_text())
print('Development material gate:',summary['material_gate_passed'])
RESULT_ZIP=OUT.parent/f'nb33_{RUN}_results.zip'
with zipfile.ZipFile(RESULT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in ('.json','.npz','.log'):z.write(p,p.relative_to(OUT))
print('RESULTS SAVED:',RESULT_ZIP)
print('Checkpoints remain separately on Drive; no submission uploaded.')
